#### Installing packages

In [32]:
# ! pip3 install llama-index
# ! pip3 install -U weviate-client
# ! pip3 install llama-index-vector-stores-weviate
# ! pip3 install python-dotenv torch sentence-transformers
# ! pip3 install llama-index-llms-ollama
# ! pip install llama-index-vector-stores-weaviate
# ! pip install llama-index-embeddings-huggingface
# ! pip install llama-index-embeddings-instructor

In [1]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.ollama import Ollama
from llama_index.core.settings import Settings
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.vector_stores.weaviate import WeaviateVectorStore

Settings.llm = Ollama(model='llama2', request_timeout=600)
Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

/home/lucas/anaconda3/envs/llm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Loading Data

Aqui, estamos carregando os dados em uma lista de objetos `Document`.

In [2]:
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader(
    input_dir="../Documents/Guidelines/",
    num_files_limit=3
).load_data()

documents

[Document(id_='4493b81c-b4ca-4e42-b6cc-9b0e0df20e80', embedding=None, metadata={'page_label': '223', 'file_name': 'ACOG-2010.pdf', 'file_path': '/home/lucas/Documents/Pessoal/Projetos/Local-LLM/llama-index/../Documents/Guidelines/ACOG-2010.pdf', 'file_type': 'application/pdf', 'file_size': 150348, 'creation_date': '2024-03-25', 'last_modified_date': '2024-03-22'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, text=' VOL. 116, NO. 1, JULY 2010  OBSTETRICS & GYNECOLOGY     223PRACTICE\nBULLETIN\nTHE AMERICAN  COLLEGE  OF OBSTETRICIANS  AND GYNECOLOGISTS\nWOMEN ’S HEALTH  CARE PHYSICIANS\nBackground\nIncidence \nEndometriosis is a gynecologic condition that occurs \nin 6–10% of women of reproductive age (2), with a \nprevalence of 38% (range, 20–50%) in inferti

In [3]:
type(documents)

list

In [3]:
documents[0].metadata['type'] = 'paper'
documents[0].metadata

{'page_label': '223',
 'file_name': 'ACOG-2010.pdf',
 'file_path': '/home/lucas/Documents/Pessoal/Projetos/Local-LLM/llama-index/../Documents/Guidelines/ACOG-2010.pdf',
 'file_type': 'application/pdf',
 'file_size': 150348,
 'creation_date': '2024-03-25',
 'last_modified_date': '2024-03-22',
 'type': 'paper'}

#### Separando os documentos em nós

Aqui nós vamos usar a técnica do context window, onde guardamos apenas uma sentença, e os textos em volta como metadados.

In [4]:
from llama_index.core.node_parser import SentenceWindowNodeParser

# creating node parser with default settings

node_parser = SentenceWindowNodeParser.from_defaults(
    window_size=5,
    window_metadata_key="window",
    original_text_metadata_key="original_text",
    include_prev_next_rel=True
)

# extracting nodes from documents
nodes = node_parser.get_nodes_from_documents(documents)

In [5]:
type(nodes)

list

In [5]:
print(nodes[3].text)
print("----------------------")
print(nodes[3].metadata['window'])

Contrary to much speculation, there are no data to 
support the view that the incidence of endometriosis is 
increasing (10), although improved recognition of endo-
metriotic lesions may have led to an increase in the rate 
of detection (11). 
----------------------
 VOL.  116, NO.  1, JULY 2010  OBSTETRICS & GYNECOLOGY     223PRACTICE
BULLETIN
THE AMERICAN  COLLEGE  OF OBSTETRICIANS  AND GYNECOLOGISTS
WOMEN ’S HEALTH  CARE PHYSICIANS
Background
Incidence 
Endometriosis is a gynecologic condition that occurs 
in 6–10% of women of reproductive age (2), with a 
prevalence of 38% (range, 20–50%) in infertile women 
(3–6), and in 71–87% of women with chronic pelvic pain 
(7–9).  Contrary to much speculation, there are no data to 
support the view that the incidence of endometriosis is 
increasing (10), although improved recognition of endo-
metriotic lesions may have led to an increase in the rate 
of detection (11).  There also appears to be no particular 
racial predisposition to endomet

In [6]:
import weaviate

client = weaviate.Client(
    embedded_options=weaviate.embedded.EmbeddedOptions()
)

print("Client is ready: ", client.is_ready())

Started /home/lucas/.cache/weaviate-embedded: process ID 1644013


{"action":"startup","default_vectorizer_module":"none","level":"info","msg":"the default vectorizer modules is set to \"none\", as a result all new schema classes without an explicit vectorizer setting, will use this vectorizer","time":"2024-04-18T20:24:21-03:00"}
{"action":"startup","auto_schema_enabled":true,"level":"info","msg":"auto schema enabled setting is set to \"true\"","time":"2024-04-18T20:24:21-03:00"}
{"level":"info","msg":"No resource limits set, weaviate will use all available memory and CPU. To limit resources, set LIMIT_RESOURCES=true","time":"2024-04-18T20:24:21-03:00"}
{"level":"warning","msg":"Multiple vector spaces are present, GraphQL Explore and REST API list objects endpoint module include params has been disabled as a result.","time":"2024-04-18T20:24:21-03:00"}
{"action":"grpc_startup","level":"info","msg":"grpc server listening at [::]:50060","time":"2024-04-18T20:24:21-03:00"}
{"action":"restapi_management","level":"info","msg":"Serving weaviate at http://12

Client is ready:  True


{"action":"read_disk_use","level":"warning","msg":"disk usage currently at 83.58%, threshold set to 80.00%","path":"/home/lucas/.local/share/weaviate","time":"2024-04-18T20:24:22-03:00"}
/home/lucas/anaconda3/envs/llm/lib/python3.10/site-packages/weaviate/warnings.py:121: DeprecationWarning: Dep005: You are using weaviate-client version 3.26.2. The latest version is 4.5.5.
            Please consider upgrading to the latest version. See https://weaviate.io/developers/weaviate/client-libraries/python for details.
  warnings.warn(


{"action":"lsm_recover_from_active_wal","class":"MyExternalContext","index":"myexternalcontext","level":"warning","msg":"active write-ahead-log found. Did weaviate crash prior to this? Trying to recover...","path":"/home/lucas/.local/share/weaviate/myexternalcontext/XiyAy9r2ER87/lsm/objects/segment-1712348607119082817.wal","shard":"XiyAy9r2ER87","time":"2024-04-18T20:24:22-03:00"}
{"action":"lsm_recover_from_active_wal_success","class":"MyExternalContext","index":"myexternalcontext","level":"info","msg":"successfully recovered from write-ahead-log","path":"/home/lucas/.local/share/weaviate/myexternalcontext/XiyAy9r2ER87/lsm/objects/segment-1712348607119082817.wal","shard":"XiyAy9r2ER87","time":"2024-04-18T20:24:22-03:00"}
{"action":"lsm_recover_from_active_wal","class":"MyExternalContext","index":"myexternalcontext","level":"warning","msg":"active write-ahead-log found. Did weaviate crash prior to this? Trying to recover...","path":"/home/lucas/.local/share/weaviate/myexternalcontext/X

In [7]:
type(client)

weaviate.client.Client

{"action":"read_disk_use","level":"warning","msg":"disk usage currently at 83.58%, threshold set to 80.00%","path":"/home/lucas/.local/share/weaviate","time":"2024-04-18T20:26:52-03:00"}


In [21]:
# RUN ONLY FOR THE FIRST TIME
index_name = "MyExternalContext"

# Constructing the vector store
vector_store = WeaviateVectorStore(
    weaviate_client=client,
    index_name=index_name
)

# Constructing the storage context
storage_context = StorageContext.from_defaults(
    vector_store=vector_store
)

# Delete duplicated index
if client.schema.exists(index_name):
    client.schema.delete_class(index_name)

# Create the index
index = VectorStoreIndex(
    nodes,
    storage_context=storage_context,
)

{"level":"info","msg":"Created shard myexternalcontext_XiyAy9r2ER87 in 3.035517ms","time":"2024-03-27T22:02:55-03:00"}
{"action":"hnsw_vector_cache_prefill","count":1000,"index_id":"main","level":"info","limit":1000000000000,"msg":"prefilled vector cache","time":"2024-03-27T22:02:55-03:00","took":69562}


In [12]:
client.

In [11]:
# Assuming 'client' is your WeaviateClient object
existing_indexes = client.get_indexes()

# Now 'existing_indexes' contains a list of all existing vector stores
for index in existing_indexes:
    print(index)

AttributeError: 'Client' object has no attribute 'get_indexes'

In [8]:
index_name = "MyExternalContext"

vector_store = WeaviateVectorStore(
    weaviate_client=client,
    index_name=index_name
)

loaded_index = VectorStoreIndex.from_vector_store(vector_store)

In [10]:
type(loaded_index)

llama_index.core.indices.vector_store.base.VectorStoreIndex

In [9]:
import json
response = client.schema.get(index_name)

print(json.dumps(response, indent=2))

{
  "class": "MyExternalContext",
  "description": "This property was generated by Weaviate's auto-schema feature on Wed Mar 27 22:02:55 2024",
  "invertedIndexConfig": {
    "bm25": {
      "b": 0.75,
      "k1": 1.2
    },
    "cleanupIntervalSeconds": 60,
    "stopwords": {
      "additions": null,
      "preset": "en",
      "removals": null
    }
  },
  "multiTenancyConfig": {
    "enabled": false
  },
  "properties": [
    {
      "dataType": [
        "text"
      ],
      "description": "This property was generated by Weaviate's auto-schema feature on Wed Mar 27 22:02:55 2024",
      "indexFilterable": true,
      "indexSearchable": true,
      "name": "file_name",
      "tokenization": "word"
    },
    {
      "dataType": [
        "uuid"
      ],
      "description": "This property was generated by Weaviate's auto-schema feature on Wed Mar 27 22:02:55 2024",
      "indexFilterable": true,
      "indexSearchable": false,
      "name": "ref_doc_id"
    },
    {
      "dataType

#### Query engine

Metadata Replacement Post Processor 

Substituindo as sentença nos nós pelos textos em volta nos metadados

In [10]:
from llama_index.core.postprocessor import MetadataReplacementPostProcessor

postproc = MetadataReplacementPostProcessor(
    target_metadata_key="window"
)

### Re-ranker

Re-rankeia o contexto retornado pela relevância com a query.

In [11]:
from llama_index.core.postprocessor import SentenceTransformerRerank

rerank = SentenceTransformerRerank(
    top_n = 2,
    model = "BAAI/bge-reranker-base"
)

### Query engine

In [12]:
query_engine = loaded_index.as_query_engine(
    similarity_top_k = 6,
    vector_store_query_mode = "hybrid",
    alpha = 0.5,
    node_postprocessor = [postproc, rerank]
)

### Running on my data

In [13]:
response = query_engine.query("I have received the diagnosis of endometriosis. What treatment options do I have? Give me a concise answer with a simple language. Also give me where in the context you found that information")
print(str(response))

Based on the provided context, there are several treatment options available for endometriosis. These include:

1. Hormonal therapies: These medications can help reduce the growth of endometrial tissue and relieve pain. Examples include hormonal birth control pills, progestins, and gonadotropin-releasing hormone agonists. (Page label: 927)
2. Surgical therapies: Laparoscopic surgery or robotic surgery can be used to remove endometrial tissue and scar tissue. (Page label: 591)
3. Alternative therapies: Acupuncture, herbal remedies, and dietary changes may also be helpful in managing endometriosis symptoms. (Page label: 228)

It is important to note that the most appropriate treatment option will depend on the severity of your symptoms, your overall health, and your personal preferences. It is recommended to consult with a healthcare provider for a proper evaluation and treatment plan. (Page label: 927)


In [15]:
response.source_nodes

[NodeWithScore(node=TextNode(id_='86c71e2a-7e46-4d1e-872a-95b0e53cabb2', embedding=[0.02203992, 0.005605236, 0.022123886, 0.052295335, -0.017546333, -0.020275028, -0.029403372, 0.07876473, -0.018428328, -0.032446247, 0.03607036, -0.07851007, 0.017186744, 0.06717063, -0.027738616, 0.05640574, -0.03149473, 0.04479067, -0.04200211, 0.07456278, 0.01766967, 0.0014005138, -0.031014454, -0.019313624, 0.005483552, 0.03666332, -0.03326387, -0.021857783, -0.03605522, -0.124392904, 0.0149655575, 0.0062794397, -0.013003561, 0.027189868, -0.032924734, -0.011071659, 0.0065270504, 0.0049103494, -0.052890096, 0.045458466, 0.09805353, 0.00850218, -0.021533601, 0.014174671, 0.034059558, 0.03855725, 0.023677802, -0.06972173, 0.03510544, 0.019741526, -0.014808755, -0.02161896, -0.0391984, 0.034420535, -0.027520446, -0.012968577, -0.0156020885, -0.015360751, -0.0060644317, 0.03037322, 0.02499434, 0.03952744, -0.17508523, 0.04255758, 0.023226295, -0.0049940906, 0.04944997, -0.0737001, -0.0064894333, 0.01327

In [16]:
window = response.source_nodes[0].node.metadata['window']
sentence = response.source_nodes[0].node.metadata['original_text']

print(f"Window: {window}")
print("------------------")
print(f"Original Sentence: {sentence}")

Window: Endometriosis and infertility:
a committee opinion
The Practice Committee of the American Society for Reproductive Medicine
American Society for Reproductive Medicine, Birmingham, Alabama
Women with endometriosis typically present with pelvic pain, infertility, or an adnexal mass, and may require surgery.  Treatment of
endometriosis in the setting of infertility raises a number of complex clinical questions that donot have simple answers.  This document replaces the 2006 ASRM Practice Committee documentof the same name.  (Fertil Steril
/C2102012;98:591– 8./C2112012 by American Society for Reproductive
Medicine.)
 Earn online CME credit related to this document at www.asrm.org/elearn
Discuss: You can discuss this article with its authors and with other ASRM members at http://
fertstertforum.com/goldsteinj-endometriosis-and-infertility-a-committee-opinion/
Use your smartphone
to scan this QR code
and connect to thediscussion forum for
this article now. *
* Download a free QR code